# Juvenile mouse skull µCT gross morphology workflow

This notebook reproduces workflow used to extract gross skull morphology measurements from the raw `.ct.zip` scan exports.

Endpoints measured:
- Skull length
- Skull width
- Skull height, full bony extent
- Skull height, cranium-only estimate
- Cranial vault length
- Cranial vault width
- Snout length

## 1. Setup

Install/import dependencies. The workflow only needs standard Python scientific/image libraries.

In [1]:
import sys
!{sys.executable} -m pip install pillow

You should consider upgrading via the 'C:\Users\mkmve\AppData\Local\Programs\Python\Python37-32\python.exe -m pip install --upgrade pip' command.


In [2]:
from pathlib import Path
import zipfile
import csv
import math
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import display

# Voxel size used in the scans.
VOXEL_MM = 0.046922

# Raw image shape used for uCT files.
DEFAULT_SHAPE = (1008, 512, 512)


## 2. Utility functions

These functions extract the CT zip, read the raw image as a memory map, create maximum-intensity projections, crop views, calculate distances, and generate annotated images/CSV output.

In [3]:
def extract_ct_zip(zip_path, extract_dir=None):
    """Extract a .ct.zip file and return the first .img/.ct.img path."""
    zip_path = Path(zip_path)
    if extract_dir is None:
        extract_dir = zip_path.with_suffix('').with_suffix('')
    extract_dir = Path(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    if not list(extract_dir.glob('*.img')):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_dir)

    img_files = list(extract_dir.glob('*.ct.img')) + list(extract_dir.glob('*.img'))
    if not img_files:
        raise FileNotFoundError(f'No .img/.ct.img file found after extracting {zip_path}')
    return img_files[0]


def make_mips(img_path, out_dir, prefix, shape=DEFAULT_SHAPE, dtype=np.int16, percentile_clip=(1, 99.7)):
    """Create MIP images along axes 0, 1, and 2."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    arr = np.memmap(img_path, dtype=dtype, mode='r', shape=shape)
    mip_paths = {}
    for axis in [0, 1, 2]:
        mip = np.max(arr, axis=axis)
        lo, hi = np.percentile(mip, percentile_clip)
        im = np.clip((mip - lo) / (hi - lo), 0, 1)
        out_path = out_dir / f'axis{axis}_{prefix}.png'
        Image.fromarray((im * 255).astype(np.uint8)).save(out_path)
        mip_paths[axis] = out_path
        del mip, im
    return mip_paths


def crop_image(image_path, crop_box, out_path):
    """Crop an image using PIL box coordinates: (left, upper, right, lower)."""
    img = Image.open(image_path)
    crop = img.crop(crop_box)
    crop.save(out_path)
    return Path(out_path)


def inspect_threshold_extents(image_path, thresholds=(55, 60, 65, 70, 75, 80), row_list=None):
    """Print rough threshold bounding boxes and row spans to guide landmark placement."""
    img = np.array(Image.open(image_path).convert('L'))
    print(image_path, img.shape)
    for thr in thresholds:
        mask = img > thr
        ys, xs = np.where(mask)
        if len(xs) == 0:
            print('thr', thr, 'no pixels')
        else:
            print('thr', thr, 'bbox', xs.min(), xs.max(), ys.min(), ys.max(),
                  'size', xs.max() - xs.min() + 1, ys.max() - ys.min() + 1)

    if row_list is not None:
        print('\nSelected row spans using thr > 70:')
        mask = img > 70
        for y in row_list:
            if 0 <= y < mask.shape[0]:
                xs = np.where(mask[y])[0]
                if len(xs):
                    print(y, xs.min(), xs.max(), xs.max() - xs.min() + 1)


def mm_from_pixels(pixel_distance, voxel_mm=VOXEL_MM):
    return float(pixel_distance) * voxel_mm


def double_arrow(draw, x1, y1, x2, y2, color, width=5):
    draw.line((x1, y1, x2, y2), fill=color, width=width)
    for (xa, ya, xb, yb) in [(x1, y1, x2, y2), (x2, y2, x1, y1)]:
        ang = math.atan2(yb - ya, xb - xa)
        ah = 10
        for off in (math.pi / 7, -math.pi / 7):
            xh = xa + ah * math.cos(ang + off)
            yh = ya + ah * math.sin(ang + off)
            draw.line((xa, ya, xh, yh), fill=color, width=width)


def dashed_line(draw, x1, y1, x2, y2, dash=10, gap=8, fill='white', width=3):
    dx = x2 - x1
    dy = y2 - y1
    dist = (dx * dx + dy * dy) ** 0.5
    t = 0.0
    while t < dist:
        s = t / dist
        e = min((t + dash) / dist, 1.0)
        xa = x1 + dx * s
        ya = y1 + dy * s
        xb = x1 + dx * e
        yb = y1 + dy * e
        draw.line((xa, ya, xb, yb), fill=fill, width=width)
        t += dash + gap


def label(draw, x, y, text, color, font):
    bbox = draw.multiline_textbbox((x, y), text, font=font, spacing=2)
    draw.rectangle((bbox[0] - 4, bbox[1] - 2, bbox[2] + 4, bbox[3] + 2), fill=(0, 0, 0))
    draw.multiline_text((x, y), text, fill=color, font=font, spacing=2)


def load_fonts():
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 20)
        font_sm = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 17)
    except Exception:
        font = ImageFont.load_default()
        font_sm = ImageFont.load_default()
    return font, font_sm


## 3. Landmark schema

The measurements are derived from landmark coordinates on two projections:

- **Dorsal projection**: skull length, skull width, cranial vault length, cranial vault width, snout length.
- **Sagittal projection**: full bony skull height and cranium-only height.

Coordinate convention: `(x, y)` in the cropped image, with `x` increasing left-to-right and `y` increasing top-to-bottom.


In [4]:
# Edit these landmark coordinates for each sample after viewing the cropped projections.
# These are starting values, but may need to be adapted for specific sample.
landmarks = {
    # Dorsal landmarks
    'x_len': 24,
    'y_top': 35,          # anterior nasal tip
    'y_boundary': 185,    # proposed anterior cranial vault boundary
    'y_bottom': 520,      # posterior skull/vault edge
    'x_width1': 38,
    'x_width2': 288,
    'y_width': 405,
    'x_vw1': 50,
    'x_vw2': 276,
    'y_vw': 360,
    'x_snout': 300,
    'x_vault': 303,

    # Sagittal landmarks
    'x_hf1': 48,
    'x_hf2': 296,
    'y_hf': 330,
    'x_hc1': 48,
    'x_hc2': 225,
    'y_hc': 330,
}


## 4. Measurement and annotation function

In [5]:
def measure_and_annotate(sample_id, dorsal_crop_path, sagittal_crop_path, landmarks, out_dir='outputs', voxel_mm=VOXEL_MM, qc_note='Pilot landmark-based measurement'):
    """Calculate endpoints, write CSV, and save annotated image."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    img_d = Image.open(dorsal_crop_path).convert('RGB')
    img_s = Image.open(sagittal_crop_path).convert('RGB')

    font, font_sm = load_fonts()

    left_x, top_y = 30, 30
    right_x = left_x + img_d.width + 20
    W = right_x + img_s.width + 30
    H = top_y + max(img_d.height, img_s.height) + 20
    canvas = Image.new('RGB', (W, H), (230, 230, 230))
    canvas.paste(img_d, (left_x, top_y))
    canvas.paste(img_s, (right_x, top_y))
    draw = ImageDraw.Draw(canvas)

    for x0, im, title in [(left_x, img_d, 'Dorsal projection: length/width/snout/vault'),
                          (right_x, img_s, 'Sagittal projection: height')]:
        bbox = draw.textbbox((0, 0), title, font=font)
        tx = x0 + (im.width - (bbox[2] - bbox[0])) // 2
        draw.text((tx, 0), title, fill='black', font=font)
        for gx in range(0, im.width + 1, 50):
            draw.line((x0 + gx, top_y, x0 + gx, top_y + im.height), fill=(150, 150, 0), width=1)
        for gy in range(0, im.height + 1, 50):
            draw.line((x0, top_y + gy, x0 + im.width, top_y + gy), fill=(150, 150, 0), width=1)

    L = landmarks

    # Pixel distances
    skull_length_px = L['y_bottom'] - L['y_top']
    skull_width_px = L['x_width2'] - L['x_width1']
    full_height_px = L['x_hf2'] - L['x_hf1']
    cranium_height_px = L['x_hc2'] - L['x_hc1']
    vault_length_px = L['y_bottom'] - L['y_boundary']
    vault_width_px = L['x_vw2'] - L['x_vw1']
    snout_length_px = L['y_boundary'] - L['y_top']

    measurements = {
        'Skull length': skull_length_px,
        'Skull width': skull_width_px,
        'Skull height - full bony extent': full_height_px,
        'Skull height - cranium only estimate': cranium_height_px,
        'Cranial vault length': vault_length_px,
        'Cranial vault width': vault_width_px,
        'Snout length': snout_length_px,
    }

    # Dorsal annotations
    double_arrow(draw, left_x + L['x_len'], top_y + L['y_top'], left_x + L['x_len'], top_y + L['y_bottom'], 'cyan', 5)
    label(draw, left_x + 45, top_y + int((L['y_top'] + L['y_bottom']) / 2), f"Skull length\n{mm_from_pixels(skull_length_px, voxel_mm):.2f} mm", 'cyan', font_sm)

    double_arrow(draw, left_x + L['x_width1'], top_y + L['y_width'], left_x + L['x_width2'], top_y + L['y_width'], 'magenta', 5)
    label(draw, left_x + 105, top_y + L['y_width'] - 34, f"Skull width {mm_from_pixels(skull_width_px, voxel_mm):.2f} mm", 'magenta', font_sm)

    double_arrow(draw, left_x + L['x_vw1'], top_y + L['y_vw'], left_x + L['x_vw2'], top_y + L['y_vw'], 'deepskyblue', 5)
    label(draw, left_x + 105, top_y + L['y_vw'] + 4, f"Vault width {mm_from_pixels(vault_width_px, voxel_mm):.2f} mm", 'deepskyblue', font_sm)

    dashed_line(draw, left_x, top_y + L['y_boundary'], left_x + img_d.width, top_y + L['y_boundary'], fill='white', width=3)
    label(draw, left_x + 14, top_y + L['y_boundary'] + 2, 'proposed anterior vault boundary', 'white', font_sm)

    double_arrow(draw, left_x + L['x_snout'], top_y + L['y_top'], left_x + L['x_snout'], top_y + L['y_boundary'], 'orange', 5)
    label(draw, left_x + max(5, L['x_snout'] - 75), top_y + int((L['y_top'] + L['y_boundary']) / 2), f"Snout\n{mm_from_pixels(snout_length_px, voxel_mm):.2f} mm", 'orange', font_sm)

    double_arrow(draw, left_x + L['x_vault'], top_y + L['y_boundary'], left_x + L['x_vault'], top_y + L['y_bottom'], 'lime', 5)
    label(draw, left_x + max(5, L['x_vault'] - 90), top_y + int((L['y_boundary'] + L['y_bottom']) / 2), f"Vault length\n{mm_from_pixels(vault_length_px, voxel_mm):.2f} mm", 'lime', font_sm)

    # Sagittal annotations
    double_arrow(draw, right_x + L['x_hc1'], top_y + L['y_hc'], right_x + L['x_hc2'], top_y + L['y_hc'], 'lime', 5)
    label(draw, right_x + 65, top_y + L['y_hc'] - 54, f"Cranium-only height\n{mm_from_pixels(cranium_height_px, voxel_mm):.2f} mm", 'lime', font_sm)

    double_arrow(draw, right_x + L['x_hf1'], top_y + L['y_hf'] + 32, right_x + L['x_hf2'], top_y + L['y_hf'] + 32, 'red', 5)
    label(draw, right_x + 140, top_y + L['y_hf'] + 5, f"Full bony height\n{mm_from_pixels(full_height_px, voxel_mm):.2f} mm", 'red', font_sm)

    label(draw, right_x + 12, top_y + 12, qc_note, 'white', font_sm)

    out_img = out_dir / f'{sample_id}_gross_morphology_annotated_pilot.png'
    canvas.save(out_img)

    endpoint_definitions = {
        'Skull length': 'Dorsal/oblique projection: anterior nasal tip to posterior skull edge',
        'Skull width': 'Dorsal/oblique projection: approximate maximum left-right bony width, likely includes lateral craniofacial structures',
        'Skull height - full bony extent': 'Sagittal/oblique projection: dorsal skull surface to most ventral bony extent; may include mandible/ventral elements',
        'Skull height - cranium only estimate': 'Sagittal/oblique projection: dorsal skull surface to estimated ventral cranial base/palate; excludes lower mandible as much as possible',
        'Cranial vault length': 'Dorsal/oblique projection: anterior cranial vault boundary to posterior skull/vault edge',
        'Cranial vault width': 'Dorsal/oblique projection: left-right braincase/calvarial width near mid-vault',
        'Snout length': 'Dorsal/oblique projection: anterior nasal tip to anterior cranial vault boundary',
    }

    out_csv = out_dir / f'{sample_id}_gross_morphology_metrics.csv'
    with open(out_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Sample_ID', 'Endpoint', 'Definition_used', 'Voxel_distance', 'Measurement_mm', 'QC_note'])
        for endpoint, px in measurements.items():
            writer.writerow([sample_id, endpoint, endpoint_definitions[endpoint], px, round(mm_from_pixels(px, voxel_mm), 3), qc_note])

    return measurements, out_img, out_csv


## 5. Run one sample

Update `sample_id` and `zip_path` for each scan. Adjust the crop boxes and landmark coordinates after inspecting the projections.

In [6]:
# Example usage
sample_id = 'B2_4'
zip_path = Path('path/to/your/sample.ct.zip')
work_dir = Path('work') / sample_id
out_dir = Path('outputs')

# Uncomment and edit after placing your file path:
# img_path = extract_ct_zip(zip_path, work_dir / 'extract')
# mip_paths = make_mips(img_path, work_dir, prefix=sample_id)
# display(Image.open(mip_paths[1]).resize((254, 504)))  # dorsal-like projection used most often
# display(Image.open(mip_paths[2]).resize((256, 504)))  # sagittal-like projection used most often


## 6. Crop views

The crop boxes varied depending on scan orientation. Common crop styles used in these datasets:

- Straighter dorsal samples: `dorsal_crop = (85, 110, 390, 600)` and `sag_crop = (70, 120, 420, 620)`
- Shifted/posterior-inclusive samples: `dorsal_crop = (70, 210, 390, 760)` and `sag_crop = (70, 220, 420, 760)`
- Larger/stronger dorsal samples: `dorsal_crop = (70, 60, 390, 610)` and `sag_crop = (60, 80, 420, 650)`

Use whichever crop fully captures the skull without clipping the anterior nasal tip or posterior skull/vault edge.

In [7]:
# Example crop and display
# dorsal_crop_path = crop_image(mip_paths[1], (70, 210, 390, 760), work_dir / f'{sample_id}_dorsal_crop.png')
# sagittal_crop_path = crop_image(mip_paths[2], (70, 220, 420, 760), work_dir / f'{sample_id}_sagittal_crop.png')
# display(Image.open(dorsal_crop_path).resize((320, 550)))
# display(Image.open(sagittal_crop_path).resize((350, 540)))

# Optional threshold summaries to help place landmarks
# inspect_threshold_extents(dorsal_crop_path, row_list=range(40, 540, 20))
# inspect_threshold_extents(sagittal_crop_path, row_list=range(40, 540, 20))


## 7. Apply landmarks and export outputs

After editing the `landmarks` dictionary above, run the measurement cell below.

In [8]:
# Example measurement run
# qc_note = 'Pilot landmark-based measurement; orientation/contrast notes go here.'
# measurements, annotated_path, csv_path = measure_and_annotate(
#     sample_id,
#     dorsal_crop_path,
#     sagittal_crop_path,
#     landmarks,
#     out_dir=out_dir,
#     voxel_mm=VOXEL_MM,
#     qc_note=qc_note
# )
# display(Image.open(annotated_path))
# print(csv_path)
# {k: round(mm_from_pixels(v), 3) for k, v in measurements.items()}


## 8. Batch template

This section is a starting point for batch processing. Because the workflow is landmark-based, full automation is not recommended unless you first develop and validate a robust registration/segmentation step. The code below loops through scan files, generates projections, and creates default crops for manual review.

In [9]:
def prepare_batch_projection_review(zip_files, output_root='batch_review', crop_style='posterior_inclusive'):
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    crop_styles = {
        'standard': {'dorsal': (85, 110, 390, 600), 'sagittal': (70, 120, 420, 620)},
        'posterior_inclusive': {'dorsal': (70, 210, 390, 760), 'sagittal': (70, 220, 420, 760)},
        'large': {'dorsal': (70, 60, 390, 610), 'sagittal': (60, 80, 420, 650)},
    }
    boxes = crop_styles[crop_style]

    review_rows = []
    for zip_file in zip_files:
        zip_file = Path(zip_file)
        sample_id = zip_file.name.replace('.zip', '').replace('.ct', '')
        sample_dir = output_root / sample_id
        img_path = extract_ct_zip(zip_file, sample_dir / 'extract')
        mip_paths = make_mips(img_path, sample_dir, prefix=sample_id)
        dorsal_crop = crop_image(mip_paths[1], boxes['dorsal'], sample_dir / f'{sample_id}_dorsal_crop.png')
        sag_crop = crop_image(mip_paths[2], boxes['sagittal'], sample_dir / f'{sample_id}_sagittal_crop.png')
        review_rows.append([sample_id, str(zip_file), str(dorsal_crop), str(sag_crop)])

    review_csv = output_root / 'projection_review_manifest.csv'
    with open(review_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Sample_ID', 'Zip_Path', 'Dorsal_Crop_Path', 'Sagittal_Crop_Path'])
        writer.writerows(review_rows)

    return review_csv

# Example:
# zip_files = list(Path('path/to/ct_zips').glob('*.zip'))
# review_manifest = prepare_batch_projection_review(zip_files, output_root='batch_review', crop_style='posterior_inclusive')
# print(review_manifest)


## Notes for final analysis

- Confirm voxel size and raw image dimensions from metadata before processing final values.
- Use the same crop/landmark logic across all samples where possible.
- Flag scans with oblique orientation, low/uneven signal, or cropped anterior/posterior boundaries.
- Treat snout length and cranial vault length as more definition-sensitive because both depend on the proposed anterior cranial vault boundary.
- Consider having landmarks placed blinded to exposure group and/or by a second rater to quantify inter-rater variability.
- These gross morphology measures are not equivalent to BMD/TMD. For BMD/TMD, a calibrated threshold/phantom-based intensity-to-density workflow is needed.
